# Data Processing Notebook

This notebook contains various data processing and analysis tasks for the Sepy2.0 project.

## Table of Contents
1. [Infusion Medications Analysis](#infusion-meds-analysis)
2. [Vent Mode Analysis](#vent-mode-analysis)
3. [Ventilator Mode Mapping Analysis](#ventilator-mode-mapping-analysis)

---


In [1]:
import os
from pathlib import Path
import pandas as pd
import glob
from typing import Set, List, Union, Optional

## [Vent Mode Analysis](#vent-mode-analysis)

This section contains vent mode unique value extraction

After extracting, a manual comparison is done with the emory data and then modified to create vent groupings


### Ventilator Mode Mapping Analysis

This section provides a comprehensive mapping analysis between the unique ventilator modes found in the Grady dataset and the corresponding entries in the Emory ventilator labels dataset (`em_vent_labels.csv`).

The analysis includes:
- Mapping of each unique vent mode to specific entries in the em_vent_labels.csv
- Recommended `vent_cat` classifications based on patterns observed in the data
- Supporting examples with line references from the em_vent_labels.csv file


In [ ]:
def extract_vent_modes_from_years(data_directory: str, year_filter = None) -> pd.DataFrame:
    """
    Find all 'vent' DSV files across year directories and extract unique vent_mode values.
    
    Args:
        data_directory: Path to directory containing year subdirectories (e.g., 2015/, 2016/, etc.)
        year_filter: Filter for years to process:
            - None: Process all years found
            - tuple (start, end): Process years in range (inclusive)
            - list: Process only specified years
    
    Returns:
        DataFrame with unique vent_mode values
    """
    
    # Convert to Path object for easier manipulation
    data_path = Path(data_directory)
    
    # Set to store all unique vent_mode values
    all_unique_modes: Set[str] = set()
    
    # Track processing info
    files_processed = 0
    years_found = []
    
    print(f"Searching for year directories in: {data_path}")
    if year_filter:
        if isinstance(year_filter, tuple):
            print(f"Year range filter: {year_filter[0]} - {year_filter[1]}")
        elif isinstance(year_filter, list):
            print(f"Specific years filter: {sorted(year_filter)}")
    
    # Look for year directories (directories with numeric names)
    all_year_dirs = [d for d in data_path.iterdir() if d.is_dir() and d.name.isdigit()]
    
    # Apply year filtering
    if year_filter is None:
        # Process all years found
        year_dirs = all_year_dirs
        print(f"Processing all years found: {sorted([d.name for d in year_dirs])}")
    elif isinstance(year_filter, tuple):
        # Process years in range (inclusive)
        start_year, end_year = year_filter
        year_dirs = [d for d in all_year_dirs if start_year <= int(d.name) <= end_year]
        print(f"Available years: {sorted([d.name for d in all_year_dirs])}")
        print(f"Years in range {start_year}-{end_year}: {sorted([d.name for d in year_dirs])}")
    elif isinstance(year_filter, list):
        # Process only specified years
        year_set = set(str(year) for year in year_filter)
        year_dirs = [d for d in all_year_dirs if d.name in year_set]
        print(f"Available years: {sorted([d.name for d in all_year_dirs])}")
        print(f"Requested years: {sorted(map(str, year_filter))}")
        print(f"Years to process: {sorted([d.name for d in year_dirs])}")
    else:
        # Invalid filter type
        print(f"⚠️  Invalid year_filter type: {type(year_filter)}. Processing all years.")
        year_dirs = all_year_dirs
    
    year_dirs.sort()
    
    if not year_dirs:
        print(f"⚠️  No year directories found in {data_path}")
        return pd.DataFrame(columns=['unique_vent_modes'])
    
    print(f"Found {len(year_dirs)} year directories: {[d.name for d in year_dirs]}")
    
    # Process each year directory
    for year_dir in year_dirs:
        year = year_dir.name
        print(f"\n📁 Processing year {year}...")
        
        # Find all files starting with 'vent' and ending with .dsv
        vent_files = list(year_dir.glob("vent*.dsv"))
        
        if not vent_files:
            print(f"   No 'vent' DSV files found in {year}")
            continue
        
        years_found.append(year)
        print(f"   Found {len(vent_files)} vent file(s): {[f.name for f in vent_files]}")
        
        # Process each vent file
        for vent_file in vent_files:
            try:
                print(f"   📄 Reading {vent_file.name}...")
                
                # Read DSV file with pipe delimiter
                df = pd.read_csv(vent_file, delimiter='|', low_memory=False)
                
                # Check if vent_mode column exists
                if 'vent_mode' not in df.columns:
                    print(f"   ⚠️  'vent_mode' column not found in {vent_file.name}")
                    print(f"   Available columns: {list(df.columns)}")
                    continue
                
                # Extract unique non-null vent_mode values
                unique_modes = df['vent_mode'].dropna().unique()
                unique_modes_clean = [str(mode).strip() for mode in unique_modes if str(mode).strip()]
                
                print(f"   ✅ Found {len(unique_modes_clean)} unique vent_mode values")
                
                # Add to our master set
                all_unique_modes.update(unique_modes_clean)
                files_processed += 1
                
            except Exception as e:
                print(f"   ❌ Error reading {vent_file.name}: {e}")
                continue
    
    # Create results summary
    print(f"\n" + "="*60)
    print(f"📊 PROCESSING SUMMARY:")
    print(f"   Years processed: {len(years_found)} ({', '.join(years_found)})")
    print(f"   Files processed: {files_processed}")
    print(f"   Total unique vent_mode values: {len(all_unique_modes)}")
    print(f"="*60)
    
    # Convert set to sorted DataFrame
    if all_unique_modes:
        unique_modes_sorted = sorted(all_unique_modes)
        result_df = pd.DataFrame({
            'unique_vent_modes': unique_modes_sorted
        })
        
        print(f"\n🎯 Unique vent_mode values found:")
        for i, mode in enumerate(unique_modes_sorted, 1):
            print(f"   {i:2d}. {mode}")
        
        return result_df
    else:
        print(f"\n⚠️  No vent_mode values found across all files")
        return pd.DataFrame(columns=['unique_vent_modes'])

# =============================================================================
# MAIN EXECUTION
# =============================================================================

# =============================================================================
# CONFIGURATION - UPDATE THESE VALUES
# =============================================================================

# TODO: Update this path to your data directory containing year subdirectories
DATA_DIRECTORY = "/hpc/group/kamaleswaranlab/GradyDataset/EMR_RAW/"  # UPDATE THIS PATH

# Year range options (choose one):
# Option 1: Process all years found in directory
YEAR_RANGE = (2015, 2022)

# Option 2: Process specific range (uncomment and modify as needed)
# YEAR_RANGE = (2015, 2020)  # Process years 2015 through 2020

# Option 3: Process specific years (uncomment and modify as needed)
# YEAR_RANGE = [2018, 2019, 2021]  # Process only these specific years

print("🔍 VENT MODE ANALYSIS")
print("="*60)

# Run the analysis
unique_vent_modes_df = extract_vent_modes_from_years(DATA_DIRECTORY, YEAR_RANGE)

# Save to CSV in current directory
output_file = "unique_vent_modes.csv"
unique_vent_modes_df.to_csv(output_file, index=False)

print(f"\n💾 Results saved to: {output_file}")
print(f"   Rows: {len(unique_vent_modes_df)}")

# Display first few results
if len(unique_vent_modes_df) > 0:
    print(f"\n📋 First 10 unique vent modes:")
    print(unique_vent_modes_df.head(10).to_string(index=False))


🔍 VENT MODE ANALYSIS
Searching for year directories in: /hpc/group/kamaleswaranlab/GradyDataset/EMR_RAW
Year range filter: 2015 - 2022
Available years: ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
Years in range 2015-2022: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
Found 8 year directories: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']

📁 Processing year 2015...
   Found 1 vent file(s): ['vent_data_2015_decomp_08022019.dsv']
   📄 Reading vent_data_2015_decomp_08022019.dsv...
   ✅ Found 10 unique vent_mode values

📁 Processing year 2016...
   Found 1 vent file(s): ['vent_data_2016_decomp_08022019.dsv']
   📄 Reading vent_data_2016_decomp_08022019.dsv...
   ✅ Found 10 unique vent_mode values

📁 Processing year 2017...
   Found 1 vent file(s): ['vent_data_2017_decomp_08022019.dsv']
   📄 Reading vent_data_2017_decomp_08022019.dsv...
   ✅ Found 11 unique vent_mode values

📁 Processing year 2018...
   Found 1 vent file(s): [

### Enhanced Ventilator Mode Mapping Analysis with vent_cat Recommendations and Examples

Based on the analysis of the em_vent_labels.csv file, here are the unique ventilator modes with their corresponding matches and **recommended vent_cat classifications**:

#### vent_cat Distribution in Dataset:
- **invasive ventilation**: 112 entries (31.5%)
- **NIPPV**: 18 entries (5.1%)
- **invasive ventilation; weaning mode**: 19 entries (5.4%)
- **home**: 18 entries (5.1%)
- **HFNC (oxygen)**: 3 entries (0.8%)
- **??** (unclassified): 11 entries (3.1%)

---

#### 1. **AC** (Assist Control)
**Matches in em_vent_labels.csv:** 37+ entries
- **Examples:**
  - `AC Pressure` (line 2) → invasive ventilation
  - `AC/CMV Volume` (line 15) → invasive ventilation
  - `"AC Pressure, SIMV Pressure"` (line 13) → invasive ventilation
- **Recommended vent_cat: `invasive ventilation`**
- **Rationale:** All AC entries are consistently classified as invasive ventilation in the dataset

#### 2. **BiLevel** 
**Matches in em_vent_labels.csv:** 5 entries
- **Examples:**
  - `Bi-Level/DuoPAP/APRV` (line 64) → invasive ventilation
  - `"Bi-Level/DuoPAP/APRV, Other: APRV PH 35, PL 0, TH 3.4, TL 0.4"` (line 67) → invasive ventilation
  - `"Bi-Level/DuoPAP/APRV, NIPPV"` (line 65) → ??
- **Recommended vent_cat: `invasive ventilation`**
- **Rationale:** Primary Bi-Level entries are invasive ventilation; combinations with NIPPV are edge cases

#### 3. **BiPAP**
**Matches in em_vent_labels.csv:** 8+ entries
- **Examples:**
  - `Other: BIPAP` (line 163) → NIPPV
  - `Other: BiPAP on standby` (line 165) → NIPPV
  - `"NIPPV, Other: Bipap"` (line 118) → NIPPV
  - `"NIPPV, Other: pt on bipap"` (line 131) → NIPPV
- **Recommended vent_cat: `NIPPV`**
- **Rationale:** BiPAP is predominantly used for non-invasive ventilation; all BiPAP entries are classified as NIPPV

#### 4. **CPAP** (Continuous Positive Airway Pressure)
**Matches in em_vent_labels.csv:** 45+ entries
- **Examples:**
  - `CPAP` (line 69) → NIPPV
  - `CPAP+PS` (line 70) → NIPPV
  - `"CPAP, non-invasive"` (line 101) → NIPPV
  - `"CPAP, Other: home unit"` (line 94) → home
  - `"CPAP, Other: SBT"` (line 89) → invasive ventilation; weaning mode
- **Recommended vent_cat: `NIPPV`**
- **Rationale:** Primary CPAP entries (lines 69, 70) are NIPPV; weaning/invasive uses are secondary applications

#### 5. **HFOV** (High-Frequency Oscillatory Ventilation)
**Matches in em_vent_labels.csv:** 3 entries
- **Examples:**
  - `HFOV` (line 103) → invasive ventilation
  - `"HFOV, Other:"` (line 104) → invasive ventilation
  - `"APV CMV, CPAP+PS, HFOV"` (line 42) → invasive ventilation
- **Recommended vent_cat: `invasive ventilation`**
- **Rationale:** HFOV is exclusively used for invasive mechanical ventilation in all documented cases

#### 6. **NIPPV** (Non-Invasive Positive Pressure Ventilation)
**Matches in em_vent_labels.csv:** 75+ entries
- **Examples:**
  - `NIPPV` (line 105) → NIPPV
  - `NIPPV AVAPS` (line 106) → NIPPV
  - `"NIPPV, Other: AVAPS"` (line 111) → NIPPV
  - `"NIPPV, Other: Home Resmed"` (line 120) → NIPPV
  - `"NIPPV, Other: home settings"` (line 130) → home
- **Recommended vent_cat: `NIPPV`**
- **Rationale:** By definition, NIPPV is non-invasive positive pressure ventilation; vast majority classified as NIPPV

#### 7. **Other (Comment)**
**Matches in em_vent_labels.csv:** 150+ entries
- **Examples:**
  - `Other: AC` (line 144) → invasive ventilation
  - `Other: AVAPS` (line 151) → NIPPV
  - `Other: Home Vent` (line 174) → home
  - `Other: AIR VO` (line 146) → HFNC (oxygen)
  - `Other: SPONT` (line 221) → ??
- **Recommended vent_cat: nan**
- **Rationale:** Requires individual assessment based on specific comment content; spans all categories. unsure for now

#### 8. **SIMV** (Synchronized Intermittent Mandatory Ventilation)
**Matches in em_vent_labels.csv:** 28 entries
- **Examples:**
  - `SIMV Pressure` (line 349) → invasive ventilation
  - `SIMV Volume` (line 350) → invasive ventilation
  - `Other: SIMV` (line 220) → invasive ventilation
  - `"AC/CMV Volume, SIMV Volume"` (line 37) → invasive ventilation
- **Recommended vent_cat: `invasive ventilation`**
- **Rationale:** SIMV is an invasive mechanical ventilation mode; all entries consistently classified as invasive ventilation

#### 9. **SiPAP**
**Matches in em_vent_labels.csv:** 1 entry
- **Examples:**
  - `SiPAP` (line 352) → invasive ventilation
- **Recommended vent_cat: `invasive ventilation`**
- **Rationale:** Limited data, but existing entry suggests invasive use (likely with tracheostomy patients)

#### 10. **Spontaneous**
**Matches in em_vent_labels.csv:** 20 entries
- **Examples:**
  - `Other: Spontaneous` (line 231) → ??
  - `"Other: Placed in spontaneous Mode by Jacquelin, Resp. Therapist"` (line 208) → invasive ventilation; weaning mode
  - `Other: spontaneous mode` (line 313) → invasive ventilation; weaning mode
  - `Other: Spontaneous breathing trial` (line 232) → invasive ventilation; weaning mode
- **Recommended vent_cat: Nan for now**
- **Rationale:** Most spontaneous breathing entries are associated with weaning from mechanical ventilation, but unsure

#### 11. **Stand-by**
**Matches in em_vent_labels.csv:** 7 entries
- **Examples:**
  - `Other: stand-by` (line 317) → (empty)
  - `Other: stby` (line 318) → (empty)
  - `Other: STBY` (line 228) → (empty)
  - `Other: BiPAP on standby` (line 165) → NIPPV
  - `Other: stby NIPPV` (line 319) → (empty)
- **Recommended vent_cat: nan for now**
- **Rationale:** Stand-by indicates ventilator is available but not actively ventilating; most entries have empty categories, but unsure

#### 12. **Volume Guarantee**
**Matches in em_vent_labels.csv:** 7 entries (as "VG")
- **Examples:**
  - `"PC, PS, SIMV, VG"` (line 336) → invasive ventilation
  - `"PSV, VG"` (line 348) → invasive ventilation
  - `"PS, SIMV, VG"` (line 346) → invasive ventilation
  - `"PC, SIMV, VG"` (line 340) → invasive ventilation
- **Recommended vent_cat: `invasive ventilation`**
- **Rationale:** Volume Guarantee is a feature of invasive mechanical ventilation; all VG entries are part of invasive modes

---

#### Summary Recommendations by Category:

##### **invasive ventilation:**
- AC, BiLevel, HFOV, SIMV, SiPAP, Volume Guarantee

##### **NIPPV:**
- BiPAP, CPAP, NIPPV

##### **invasive ventilation; weaning mode:**
- Spontaneous

##### **standby/inactive** (new category suggested):
- Stand-by

##### **mixed/context-dependent:**
- Other (Comment) - requires individual assessment

#### Notes:
1. **Line references** provide specific evidence from the em_vent_labels.csv file
2. Some modes can function in multiple contexts (e.g., CPAP for both NIPPV and weaning)
3. Consider creating a "standby/inactive" category for stand-by modes
4. "Other (Comment)" entries require case-by-case evaluation based on specific comments
5. The classification reflects the predominant use pattern in the dataset with supporting examples


## [Infusion Medications Analysis](#infusion-meds-analysis)

This section contains infusion medication unique combination extraction

After extracting unique combinations of formulary_name, med_name_generic, med_action_dose_unit, volume_given, count, volume_unit, and volume_numeric, a manual comparison is done with the emory data and then modified to create medication groupings


### Infusion Medications Combination Analysis

This section provides a comprehensive extraction of unique medication combinations found in the Grady dataset infusion files.

The analysis includes:
- Extraction of unique combinations across multiple medication-related columns
- Comprehensive processing of all infusion files across multiple years
- Output of structured CSV data for further analysis and mapping


In [24]:
def extract_infusion_meds_from_years(data_directory: str, year_filter = None) -> pd.DataFrame:
    """
    Find all 'infusion' DSV files across year directories and extract unique medication combinations.
    Maps Grady dataset columns to Emory format and adds occurrence counts.
    
    Args:
        data_directory: Path to directory containing year subdirectories (e.g., 2015/, 2016/, etc.)
        year_filter: Filter for years to process:
            - None: Process all years found
            - tuple (start, end): Process years in range (inclusive)
            - list: Process only specified years
    
    Returns:
        DataFrame with unique medication combinations and counts
    """
    
    # Convert to Path object for easier manipulation
    data_path = Path(data_directory)
    
    # Column mapping from Grady to Emory format
    # Note: med_order_dose maps to both volume_given and volume_numeric
    grady_to_emory_mapping = [
        ('med_name', 'formulary_name'),
        ('med_name_generic', 'med_name_generic'), 
        ('med_action_dose_unit', 'med_action_dose_unit'),
        ('med_order_dose', 'volume_given'),  # does not have units like ml or g
        ('med_order_dose_unit', 'volume_unit'),
        ('med_order_dose', 'volume_numeric'),  # same source, different target
    ]
    
    # Grady columns to extract (source columns) - get unique ones
    grady_columns = list(dict.fromkeys([mapping[0] for mapping in grady_to_emory_mapping]))
    
    # Emory format columns (target columns) - all target columns
    emory_columns = [mapping[1] for mapping in grady_to_emory_mapping]
    
    # Final output columns (including count)
    output_columns = emory_columns + ['count']
    
    # List to store all combinations with counts
    all_combinations = []
    
    # Track processing info
    files_processed = 0
    years_found = []
    
    print(f"🔍 INFUSION MEDICATIONS ANALYSIS")
    print("="*60)
    print(f"Searching for year directories in: {data_path}")
    if year_filter:
        if isinstance(year_filter, tuple):
            print(f"Year range filter: {year_filter[0]} - {year_filter[1]}")
        elif isinstance(year_filter, list):
            print(f"Specific years filter: {sorted(year_filter)}")
    
    # Look for year directories (directories with numeric names)
    all_year_dirs = [d for d in data_path.iterdir() if d.is_dir() and d.name.isdigit()]
    
    # Apply year filtering
    if year_filter is None:
        year_dirs = all_year_dirs
        print(f"Processing all years found: {sorted([d.name for d in year_dirs])}")
    elif isinstance(year_filter, tuple):
        start_year, end_year = year_filter
        year_dirs = [d for d in all_year_dirs if start_year <= int(d.name) <= end_year]
        print(f"Available years: {sorted([d.name for d in all_year_dirs])}")
        print(f"Years in range {start_year}-{end_year}: {sorted([d.name for d in year_dirs])}")
    elif isinstance(year_filter, list):
        year_set = set(str(year) for year in year_filter)
        year_dirs = [d for d in all_year_dirs if d.name in year_set]
        print(f"Available years: {sorted([d.name for d in all_year_dirs])}")
        print(f"Requested years: {sorted(map(str, year_filter))}")
        print(f"Years to process: {sorted([d.name for d in year_dirs])}")
    else:
        print(f"⚠️  Invalid year_filter type: {type(year_filter)}. Processing all years.")
        year_dirs = all_year_dirs
    
    year_dirs.sort()
    
    if not year_dirs:
        print(f"⚠️  No year directories found in {data_path}")
        return pd.DataFrame(columns=grady_columns)
    
    print(f"Found {len(year_dirs)} year directories: {[d.name for d in year_dirs]}")
    
    # Process each year directory
    for year_dir in year_dirs:
        year = year_dir.name
        print(f"\\n📁 Processing year {year}...")
        
        # Find all files starting with 'infusion' and ending with .dsv
        infusion_files = list(year_dir.glob("infusion*.dsv"))
        
        if not infusion_files:
            print(f"   No 'infusion' DSV files found in {year}")
            continue
        
        years_found.append(year)
        print(f"   Found {len(infusion_files)} infusion file(s): {[f.name for f in infusion_files]}")
        
        # Process each infusion file
        for infusion_file in infusion_files:
            try:
                print(f"   📄 Reading {infusion_file.name}...")
                
                # Read DSV file with pipe delimiter
                df = pd.read_csv(infusion_file, delimiter='|', low_memory=False)
                
                # Check if all Grady source columns exist
                missing_columns = [col for col in grady_columns if col not in df.columns]
                if missing_columns:
                    print(f"   ⚠️  Missing columns in {infusion_file.name}: {missing_columns}")
                    print(f"   Available columns: {list(df.columns)}")
                    continue
                
                # Extract and map columns from Grady to Emory format
                # Keep NaN values as they are valid for combination grouping
                df_selected = df[grady_columns]
                
                # Create mapped DataFrame with proper column duplication
                df_mapped = pd.DataFrame()
                for grady_col, emory_col in grady_to_emory_mapping:
                    df_mapped[emory_col] = df_selected[grady_col]
                
                # Get value counts for each unique combination
                combination_counts = df_mapped.value_counts().reset_index()
                combination_counts.columns = emory_columns + ['count']
                
                print(f"   ✅ Found {len(combination_counts)} unique medication combinations")
                
                # Add to our master list
                all_combinations.append(combination_counts)
                files_processed += 1
                
            except Exception as e:
                print(f"   ❌ Error reading {infusion_file.name}: {e}")
                continue
    
    # Combine all combinations and aggregate counts
    if all_combinations:
        combined_df = pd.concat(all_combinations, ignore_index=True)
        
        # Group by all medication columns and sum the counts
        final_unique_df = combined_df.groupby(emory_columns, as_index=False)['count'].sum()
        
        # Sort by formulary_name for better organization
        final_unique_df = final_unique_df.sort_values('formulary_name').reset_index(drop=True)
    else:
        final_unique_df = pd.DataFrame(columns=output_columns)
    
    # Create results summary
    print(f"\\n" + "="*60)
    print(f"📊 PROCESSING SUMMARY:")
    print(f"   Years processed: {len(years_found)} ({', '.join(years_found)})")
    print(f"   Files processed: {files_processed}")
    print(f"   Total unique medication combinations: {len(final_unique_df)}")
    print(f"="*60)
    
    if len(final_unique_df) > 0:
        print(f"\\n🎯 Sample medication combinations found:")
        sample_size = min(5, len(final_unique_df))
        for i in range(sample_size):
            row = final_unique_df.iloc[i]
            print(f"   {i+1:2d}. {row['formulary_name']} | {row['med_name_generic']} | {row['volume_given']} | Count: {row['count']}")
        
        return final_unique_df
    else:
        print(f"\\n⚠️  No medication combinations found across all files")
        return pd.DataFrame(columns=output_columns)

# =============================================================================
# INFUSION MEDICATIONS ANALYSIS EXECUTION
# =============================================================================

# Configuration - using same data directory and year range as vent analysis
DATA_DIRECTORY = "/hpc/group/kamaleswaranlab/GradyDataset/EMR_RAW/"
YEAR_RANGE = (2015, 2022)

# Run the infusion medications analysis
unique_infusion_meds_df = extract_infusion_meds_from_years(DATA_DIRECTORY, YEAR_RANGE)

# Save to CSV in current directory
output_file = "unique_infusion_medication_combinations.csv"
unique_infusion_meds_df.to_csv(output_file, index=False)

print(f"\\n💾 Results saved to: {output_file}")
print(f"   Rows: {len(unique_infusion_meds_df)}")

# Display first few results
if len(unique_infusion_meds_df) > 0:
    print(f"\\n📋 First 10 unique medication combinations:")
    print(unique_infusion_meds_df.head(10).to_string(index=False))


🔍 INFUSION MEDICATIONS ANALYSIS
Searching for year directories in: /hpc/group/kamaleswaranlab/GradyDataset/EMR_RAW
Year range filter: 2015 - 2022
Available years: ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
Years in range 2015-2022: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
Found 8 year directories: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022']
\n📁 Processing year 2015...
   Found 1 infusion file(s): ['infusion_meds_2015_decomp_03112020.dsv']
   📄 Reading infusion_meds_2015_decomp_03112020.dsv...
   ✅ Found 925 unique medication combinations
\n📁 Processing year 2016...
   Found 1 infusion file(s): ['infusion_meds_2016_decomp_03112020.dsv']
   📄 Reading infusion_meds_2016_decomp_03112020.dsv...
   ✅ Found 1023 unique medication combinations
\n📁 Processing year 2017...
   Found 1 infusion file(s): ['infusion_meds_2017_decomp_03112020.dsv']
   📄 Reading infusion_meds_2017_decomp_03112020.dsv...
   ✅ Found 1179 unique m